# Demo he thong goi y san pham tu Clickstream

Notebook nay goi truc tiep vao API cua he thong (dang chay bang `docker compose up`) de minh hoa toan bo pipeline:

`Browser/Notebook -> Flask API -> Kafka -> Spark Streaming -> Markov Chain / GRU4Rec -> Redis -> API tra ve`

**Yeu cau truoc khi chay:** he thong dang chay o `http://localhost:5000` (kiem tra bang `docker compose ps`).

In [ ]:
import requests
import time
import subprocess
import json

API_URL = "http://localhost:5000"
print("API san sang:", requests.get(API_URL + "/api/recommend/healthcheck").status_code in (200,))

## Buoc 1 - Xem quy mo du lieu that dang co trong he thong

Du lieu la tap RetailRocket e-commerce that (khong phai gia lap), da nap vao Cassandra.

In [ ]:
result = subprocess.run(
    ["docker", "exec", "clickstream-basedrecommendationsystem-cassandra-1",
     "cqlsh", "-e", "SELECT COUNT(*) FROM recsys.events;"],
    capture_output=True, text=True,
)
print(result.stdout)

## Buoc 2 - Gia lap mot nguoi dung xem san pham

Thay vi click chuot tren web, ta goi thang API `/api/track` - day la endpoint ma frontend goi khi nguoi dung xem/them vao gio mot san pham.

In [ ]:
session_id = "notebook-demo"
item_id = "63311"  # mot item_id that trong dataset RetailRocket

payload = {
    "session_id": session_id,
    "item_id": item_id,
    "event_type": "view",
    "timestamp": int(time.time() * 1000),
}

resp = requests.post(f"{API_URL}/api/track", json=payload)
print("Gui event thanh cong:", resp.status_code, resp.json())

## Buoc 3 - Doi Spark Streaming xu ly (vai giay)

Event vua gui di qua Kafka, Spark Structured Streaming doc theo micro-batch, chay inference 2 model (Markov Chain + GRU4Rec) roi luu ket qua vao Redis.

In [ ]:
time.sleep(8)

resp = requests.get(f"{API_URL}/api/recommend/{session_id}")
reco = resp.json()

print("Goi y tu Markov Chain:", reco["markov"])
print("Goi y tu GRU4Rec   :", reco["gru4rec"])

## Buoc 4 - Mo phong mot phien duyet web: xem lien tiep nhieu san pham

Goi y se thay doi theo tung buoc vi GRU4Rec dung toan bo ngu canh phien (session), con Markov Chain chi dung item vua xem gan nhat.

In [ ]:
session_id = "notebook-session-2"
items_xem_lien_tiep = ["63311", "24165", "42819"]  # gia su user xem lan luot 3 san pham

for item_id in items_xem_lien_tiep:
    requests.post(f"{API_URL}/api/track", json={
        "session_id": session_id,
        "item_id": item_id,
        "event_type": "view",
        "timestamp": int(time.time() * 1000),
    })
    time.sleep(6)

    reco = requests.get(f"{API_URL}/api/recommend/{session_id}").json()
    print(f"Sau khi xem {item_id}:")
    print("  Markov :", reco["markov"])
    print("  GRU4Rec:", reco["gru4rec"])
    print()

## Ket luan

- Pipeline end-to-end (API -> Kafka -> Spark Streaming -> model -> Redis -> API) hoat dong dung tren du lieu RetailRocket that.
- Markov Chain phan hoi theo item gan nhat (suy luan nhanh, don gian).
- GRU4Rec phan hoi theo toan bo chuoi hanh vi trong phien (nham bat ngu canh dai hon).